# 1. Define an index, calculate it, backtest a tracker

The smallest complete pass through all three layers of the library:

> **methodology** → **calculator** → **backtest**

Everything here is generated. No network, no data files, nothing prepared —
run the cells top to bottom on a clean checkout and it works.

## The one thing to notice

The index level and the backtest NAV are **not the same number**, even at zero
transaction cost. The calculator computes a market-cap-style level
(price × shares ÷ divisor); the engine holds a weight-rebalanced portfolio that
drifts between rebalances and trades in whole amounts of cash.

They track closely and diverge slightly, and that divergence is exactly what a
real tracking portfolio experiences. The last section measures it.

## Setup

Only the core library and pandas — no optional extras.

In [ ]:
import logging

import pandas as pd

from beacon.backtest.engine import BacktestEngine
from beacon.index.calculation import IndexCalculator
from beacon.index.constructor import IndexDefinition
from beacon.index.methodology import MarketCapWeighted
from beacon.synthetic import SyntheticConfig, generate

# The engine warns once per rebalance when a buy cannot be fully funded --
# real behaviour worth knowing about, but seventeen lines of it buries the
# output a notebook exists to show. Drop this to WARNING to watch them happen.
logging.basicConfig(level=logging.ERROR,
                    format="%(levelname)s %(name)s: %(message)s")

pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## Step 1 — generate a market

`generate()` builds a universe, simulates correlated returns through a
GJR-GARCH process, turns them into OHLCV prices, and applies splits and
dividends. It returns everything a `DataFetcher` needs.

In [ ]:
# Small enough to run in seconds, large enough that the cap actually binds.
#
# The seed is chosen, not arbitrary. It has to clear two bars: the index must
# not *lose* money over four years (a demo that does reads as a broken library
# rather than as an unlucky path), and the 10% cap has to bite on more than one
# name without being absurd. On this seed the largest name reaches 14.9%
# before capping and two names are capped, moving 6% of the index. Nothing
# else is tuned - change the seed and you get a different, equally valid
# market.
CONFIG = SyntheticConfig(assets=40,
                         start="2021-01-04",
                         end="2024-12-31",
                         seed=3)

dataset = generate(CONFIG)
fetcher = dataset.fetcher()

pd.Series({"identifiers": len(dataset.universe),
           "business days": len(dataset.returns),
           "corporate actions": len(dataset.actions.data),
           "first date": CONFIG.start,
           "last date": CONFIG.end},
          name="dataset")

The universe carries the reference data an index definition selects on:

In [ ]:
dataset.universe.head(10)

## Step 2 — define the methodology

An `IndexDefinition` is **static rules only**. It holds no prices and computes
nothing: what the universe is, how names are weighted, how often it rebalances,
and the cap that no single name may exceed.

Separating the rules from the calculation is what lets the same definition be
run over different data, or the same data under different rules.

In [ ]:
definition = IndexDefinition(
    index_id="DEMO",
    index_name="DEMO Index",
    base_date=CONFIG.start,
    base_value=1000.0,
    currency=CONFIG.currency,
    eligibility_rules=[],
    weighting_scheme=MarketCapWeighted(use_free_float=True),
    rebalancing_frequency="QUARTERLY",
    universe_identifiers=list(dataset.universe.index),
    max_constituent_weight=0.10)

pd.Series({"weighting": definition.weighting_scheme.scheme_name,
           "rebalances": definition.rebalancing_frequency,
           "cap": f"{definition.max_constituent_weight:.0%}",
           "universe": f"{len(definition.universe_identifiers)} names",
           "base": f"{definition.base_value:,.0f} on {CONFIG.start}"},
          name="methodology")

## Step 3 — calculate the index

`IndexCalculator.run()` walks business days, applies the weighting scheme at
each rebalance, and maintains the **divisor** — the number that absorbs
composition changes so the level moves on performance alone.

In [ ]:
index = IndexCalculator(definition, fetcher).run(start_date=CONFIG.start,
                                                 end_date=CONFIG.end)

levels = index.index_levels

pd.Series({"base level": levels.iloc[0],
           "final level": levels.iloc[-1],
           "total return": f"{levels.iloc[-1] / levels.iloc[0] - 1:.2%}",
           "observations": len(levels),
           "rebalances": len(index.weight_snapshots)},
          name="index")

The level series itself:

In [ ]:
levels.head()

The divisor is the mechanism that makes the level continuous. When a
constituent changes, the market value jumps; the divisor is reset so the level
does not.

In [ ]:
index.divisor_history.head()

### The composition in force at the end of the run

In [ ]:
latest = max(index.weight_snapshots)
weights = pd.Series(index.weight_snapshots[latest], name="weight")

weights.sort_values(ascending=False).head(10).to_frame().style.format("{:.2%}")

### Did the cap bind?

`max_constituent_weight` is applied iteratively: cap the offenders, redistribute
the excess to everyone else, and check again — because redistribution can push
a name that was previously under the limit over it.

`uncapped_weights` is the counterfactual, so you can see what the cap actually
cost each name.

In [ ]:
if index.cap_reports:
    date = max(index.cap_reports)
    report = index.cap_reports[date]

    print(f"the cap bound at {len(index.cap_reports)} rebalance(s)")
    print(f"at {date.date()} it moved {report.redistributed:.2%} "
          f"off {len(report.capped)} name(s)")

    display(pd.DataFrame({"uncapped": pd.Series(report.uncapped_weights),
                          "applied": weights})
            .dropna()
            .assign(moved=lambda frame: frame["applied"] - frame["uncapped"])
            .sort_values("moved")
            .head(8)
            .style.format("{:.2%}"))
else:
    print("the cap never bound: no name reached 10% at any rebalance")

## Step 4 — backtest a portfolio that tracks it

The engine takes the index result as a **target weight schedule** and simulates
holding it: selling before buying so cash never goes negative, and paying costs
in basis points on traded notional.

In [ ]:
CAPITAL = 10_000_000.0
COSTS_BPS = 10.0

backtest = BacktestEngine(start_date=CONFIG.start,
                          end_date=CONFIG.end,
                          initial_capital=CAPITAL,
                          data_provider=fetcher,
                          target_index_result=index,
                          transaction_cost_bps=COSTS_BPS).run()

nav = backtest.portfolio_nav

pd.Series({"initial": f"{backtest.initial_capital:,.2f}",
           "final NAV": f"{nav.iloc[-1]:,.2f}",
           "trades": f"{len(backtest.transactions):,}",
           "total costs": f"{sum(t.transaction_cost for t in backtest.transactions):,.2f}"},
          name="backtest")

In [ ]:
AS_PERCENT = {"total_return", "annualised_return", "volatility",
              "max_drawdown", "tracking_error", "tracking_difference"}

pd.Series({name: ("n/a" if value is None
                  else f"{value:.2%}" if name in AS_PERCENT
                  else f"{value:.3f}")
           for name, value in backtest.summary().items()},
          name="summary")

The individual trades, so the simulation is inspectable rather than opaque:

In [ ]:
pd.DataFrame([{"date": t.transaction_date,
               "asset": t.asset_id,
               "side": t.transaction_type,
               "quantity": t.quantity,
               "price": t.price,
               "cost": t.transaction_cost}
              for t in backtest.transactions[:8]])

## Step 5 — the gap between the two

Here is the number this notebook exists to show.

In [ ]:
index_return = levels.iloc[-1] / levels.iloc[0] - 1
portfolio_return = nav.iloc[-1] / backtest.initial_capital - 1

pd.Series({"index": f"{index_return:.2%}",
           "portfolio": f"{portfolio_return:.2%}",
           "difference": f"{portfolio_return - index_return:+.2%}"},
          name="over the whole run")

Rebased to 100, sampled quarterly, so you can watch the gap open rather than
just read its final value:

In [ ]:
comparison = pd.DataFrame({
    "index": levels / levels.iloc[0] * 100,
    "portfolio": nav / nav.iloc[0] * 100,
}).dropna()

comparison["gap"] = comparison["portfolio"] - comparison["index"]
comparison.iloc[::63]

### Why they differ

These differ **by construction, not by mistake**:

- the index is a divisor-based level, recomputed from prices and shares every
  day;
- the portfolio holds *units*, so its weights drift with relative performance
  between rebalances;
- the portfolio pays to trade, and trades in whole amounts of cash.

Costs explain part of the gap. The different construction explains the rest —
and at zero costs the gap would shrink but not vanish. The two coincide exactly
only when every price path is proportional, which no real market is.

## Where to go next

- **`02_backtest_analysis.ipynb`** — statistics, concentration, held versus
  target weights, attribution, and charts
- **`04_optimised_index.ipynb`** — replace the cap with real constraints and an
  optimiser